# 03: Model Training
Train 5 models with class balancing and 5-fold CV

In [ ]:
import pandas as pd, numpy as np, os, pickle, warnings, random
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_class_weight
import joblib
print('=== NOTEBOOK 03: MODEL TRAINING (6 TECHNIQUES) ===')
data_dir = '../data/processed'
X_train = pd.read_csv(f'{data_dir}/X_train.csv')
X_test = pd.read_csv(f'{data_dir}/X_test.csv')
y_train = pd.read_csv(f'{data_dir}/y_train.csv').squeeze()
y_test = pd.read_csv(f'{data_dir}/y_test.csv').squeeze()
label_map = pd.read_csv(f'{data_dir}/label_map.csv')
for col in X_train.select_dtypes(include='object').columns:
    le = LabelEncoder()
    all_vals = pd.concat([X_train[col], X_test[col]]).astype(str)
    le.fit(all_vals)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
X_train = X_train.fillna(0); X_test = X_test.fillna(0)
actual_classes = sorted(y_test.unique())
target_names = [label_map.loc[label_map['encoded'] == i, 'technique'].values[0] for i in actual_classes]
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Classes: {target_names}')
print(f'Class distribution (train):\n{y_train.value_counts().sort_index()}')
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print(f'\nClass weights: {class_weight_dict}')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(max_depth=20, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, eval_metric='mlogloss', verbosity=0),
    'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=RANDOM_STATE, early_stopping=True, validation_fraction=0.1)
}
all_results = []; all_predictions = {}
for name, model in models.items():
    print(f'\n=== {name.upper()} ===')
    X_tr = X_train_scaled if name in ['Logistic Regression', 'Neural Network (MLP)'] else X_train
    X_te = X_test_scaled if name in ['Logistic Regression', 'Neural Network (MLP)'] else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    all_predictions[name] = y_pred
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    print(f'Accuracy: {acc:.4f} | Precision: {pre:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')
    print(classification_report(y_test, y_pred, labels=actual_classes, target_names=target_names, zero_division=0))
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_f1 = cross_val_score(model, X_tr, y_train, cv=cv, scoring='f1_weighted')
    print(f'CV F1 (5-fold): {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
    all_results.append({'Model': name, 'Accuracy': acc, 'Precision': pre, 'Recall': rec, 'F1 Score': f1, 'CV_F1_Mean': cv_f1.mean(), 'CV_F1_Std': cv_f1.std()})
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    with open(f'../models/{safe_name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f'Saved: models/{safe_name}.pkl')
results_df = pd.DataFrame(all_results)
results_df.to_csv('../results/model_comparison.csv', index=False)
print('\n=== FINAL COMPARISON ===')
print(results_df.to_string(index=False))
pred_df = pd.DataFrame({'y_true': y_test.values, **{k: v for k, v in all_predictions.items()}})
pred_df.to_csv('../results/predictions.csv', index=False)
rf_model = models['Random Forest']; xgb_model = models['XGBoost']
fi_df = pd.DataFrame({'feature': X_train.columns, 'importance_rf': rf_model.feature_importances_, 'importance_xgb': xgb_model.feature_importances_}).sort_values('importance_rf', ascending=False)
fi_df.to_csv('../results/feature_importance.csv', index=False)
print('\n=== TRAINING COMPLETE ===')
